In [1]:
pip install requests pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENWEATHER_API_KEY")

print("Key loaded:", API_KEY is not None)  # should print True

Key loaded: True


In [4]:
import requests

city = "Nairobi"
url = "https://api.openweathermap.org/data/2.5/weather"

params = {
    "q": city,
    "appid": API_KEY,
    "units": "metric"   # gives temp in °C instead of Kelvin
}

response = requests.get(url, params=params)
print(response.status_code)
data = response.json()
data

200


{'coord': {'lon': 36.8167, 'lat': -1.2833},
 'weather': [{'id': 801,
   'main': 'Clouds',
   'description': 'few clouds',
   'icon': '02d'}],
 'base': 'stations',
 'main': {'temp': 20.62,
  'feels_like': 20.02,
  'temp_min': 20.62,
  'temp_max': 20.62,
  'pressure': 1020,
  'humidity': 49,
  'sea_level': 1020,
  'grnd_level': 843},
 'visibility': 10000,
 'wind': {'speed': 3.09, 'deg': 170},
 'clouds': {'all': 24},
 'dt': 1787822131,
 'sys': {'type': 1,
  'id': 2543,
  'country': 'KE',
  'sunrise': 1787801527,
  'sunset': 1787845022},
 'timezone': 10800,
 'id': 184745,
 'name': 'Nairobi',
 'cod': 200}

In [5]:
from datetime import datetime

def extract_weather(data):
    return {
        "city": data["name"],
        "temperature_c": data["main"]["temp"],
        "humidity_pct": data["main"]["humidity"],
        "condition": data["weather"][0]["description"],
        "wind_speed_mps": data["wind"]["speed"],
        "datetime": datetime.fromtimestamp(data["dt"])
    }

nairobi_weather = extract_weather(data)
nairobi_weather

{'city': 'Nairobi',
 'temperature_c': 20.62,
 'humidity_pct': 49,
 'condition': 'few clouds',
 'wind_speed_mps': 3.09,
 'datetime': datetime.datetime(2026, 8, 27, 12, 15, 31)}

In [6]:
cities = ["Nairobi", "London", "New York"]

weather_records = []

for city in cities:
    params = {
        "q" : city,
        "appid" : API_KEY,
        "units" : "metric"
    }
    response = requests.get(url, params=params)

    if response.status_code == 200:
        city_data = response.json()
        weather_records.append(extract_weather(city_data))
        print(f"{city} fetched successfully")
    else:
        print(f"Failed to fetch {city} - status code: {response.status_code}")

weather_records

Nairobi fetched successfully
London fetched successfully
New York fetched successfully


[{'city': 'Nairobi',
  'temperature_c': 20.93,
  'humidity_pct': 56,
  'condition': 'few clouds',
  'wind_speed_mps': 6.69,
  'datetime': datetime.datetime(2026, 8, 27, 12, 31, 30)},
 {'city': 'London',
  'temperature_c': 19.16,
  'humidity_pct': 92,
  'condition': 'overcast clouds',
  'wind_speed_mps': 2.06,
  'datetime': datetime.datetime(2026, 8, 27, 12, 34, 28)},
 {'city': 'New York',
  'temperature_c': 22.3,
  'humidity_pct': 85,
  'condition': 'scattered clouds',
  'wind_speed_mps': 3.13,
  'datetime': datetime.datetime(2026, 8, 27, 12, 30, 3)}]

## Task 1: Extract — Summary

**Goal:** Pull live weather data for 3 cities from the OpenWeather API.

### What we did

1. **Secured the API key**
   - Stored the key in a `.env` file (not hardcoded in the notebook).
   - Loaded it with `python-dotenv` so it never appears in the code itself.
   - This matters because a public GitHub repo with a raw API key gets scraped and abused — `.env` is excluded via `.gitignore`.

2. **Made a single test call**
   - Hit OpenWeather's `/data/2.5/weather` endpoint for one city (Nairobi) to inspect the raw JSON response before building anything bigger.
   - Used `units=metric` in the request so temperature comes back in °C rather than Kelvin.

3. **Mapped the JSON structure**
   - The raw response is deeply nested (e.g. temperature is inside `data['main']['temp']`, condition is inside a list `data['weather'][0]['description']`).
   - Wrote an `extract_weather()` function that pulls out just the fields we need and flattens them into one simple dictionary per city:
     - `city`, `temperature_c`, `humidity_pct`, `condition`, `wind_speed_mps`, `datetime`
   - Converted the API's Unix timestamp (`dt`) into a readable Python `datetime` object.

4. **Looped over all 3 cities**
   - Nairobi, London, New York.
   - Each city's data is fetched, extracted, and appended to a list called `weather_records`.
   - Added success/failure print statements so a failed request for one city doesn't silently break the pipeline — we'd see it immediately.

### Output

A list of 3 clean dictionaries — one per city — ready to be converted into a structured table.

### Why it matters

This is the **Extract** step of ETL: getting raw data out of a source system (an API here) reliably, with visibility into what succeeded or failed, before any cleaning or analysis happens.

In [7]:
import pandas as pd
df = pd.DataFrame(weather_records)
df

,city,temperature_c,humidity_pct,condition,wind_speed_mps,datetime
0,Nairobi,20.93,56,few clouds,6.69,2026-08-27 12:31:30
1,London,19.16,92,overcast clouds,2.06,2026-08-27 12:34:28
2,New York,22.30,85,scattered clouds,3.13,2026-08-27 12:30:03


In [8]:
df.dtypes

city                      object
temperature_c            float64
humidity_pct               int64
condition                 object
wind_speed_mps           float64
datetime          datetime64[ns]
dtype: object

In [9]:
df = df.rename(columns={
    "temperature_c" : "Temperature(°C)",
    "humidity_pct" : "Humidity(%)",
    "wind_speed_mps" : "Wind Speed (m/s)",
    "city" : "City",
    "condition" : "Weather Condition",
    "datetime" : "Date & Time"
})

df

,City,Temperature(°C),Humidity(%),Weather Condition,Wind Speed (m/s),Date & Time
0,Nairobi,20.93,56,few clouds,6.69,2026-08-27 12:31:30
1,London,19.16,92,overcast clouds,2.06,2026-08-27 12:34:28
2,New York,22.30,85,scattered clouds,3.13,2026-08-27 12:30:03


In [10]:
import os
# Make sure the data folder exists
os.makedirs("data", exist_ok=True)

df.to_csv("data/weather_data.csv", index=False)
print("Data saved to data/weather data.csv")

Data saved to data/weather data.csv


In [13]:
df.columns.tolist()

['City',
 'Temperature(°C)',
 'Humidity(%)',
 'Weather Condition',
 'Wind Speed (m/s)',
 'Date & Time']

In [17]:
# Which city is warmest / coolest?
warmest = df.loc[df["Temperature(°C)"].idxmax()]
coolest = df.loc[df["Temperature(°C)"].idxmin()]

print(f"Warmest city: {warmest['City']} at {warmest['Temperature(°C)']}°C")
print(f"Coolest city: {coolest['City']} at {coolest['Temperature(°C)']}°C")

# Which city is most humid?
most_humid = df.loc[df["Humidity(%)"].idxmax()]
print(f"💧 Most humid city: {most_humid['City']} at {most_humid['Humidity(%)']}%")

# Weather conditions side by side
print("\n Weather conditions:")
print(df[["City", "Weather Condition"]].to_string(index=False))

Warmest city: New York at 22.3°C
Coolest city: London at 19.16°C
💧 Most humid city: London at 92%

 Weather conditions:
    City Weather Condition
 Nairobi        few clouds
  London   overcast clouds
New York  scattered clouds


## Task 4: Findings Summary

On 27 August 2026, weather was compared across three cities: Nairobi, London, and New York.

- **New York** was the warmest of the three at 22.3°C, while **London** was the coolest at 19.16°C — a gap of just over 3°C despite the very different geography.
- **London** also had the highest humidity at 92%, notably higher than Nairobi (56%) and New York (85%), which lines up with its overcast sky conditions.
- Conditions varied across all three: Nairobi had few clouds, London was overcast, and New York had scattered clouds — none of the three cities recorded clear skies or rain at the time of the snapshot.
- Interestingly, Nairobi's tropical highland climate produced the lowest humidity despite not being the coolest city, showing that temperature and humidity don't always move together.